# GDP Preprocessing — Nigeria (NGA) 2024

Two extraction strategies from the Kummu v4 dataset:
- **Windowed** (`preprocess_kummu_gdp_tif_windowed`): memory-light, recommended for large global rasters
- **General extractor** (`extract_country_gdp`): auto-downloads boundary, clips, saves

In [ ]:
import os, sys
ROOT_DIR = os.path.abspath(os.curdir)
sys.path.insert(0, ROOT_DIR)

from preprocess_kummu_gdp import download_file, preprocess_kummu_gdp_tif_windowed, KUMMU_V4_FILES
from extract_gdp import extract_country_gdp
import geopandas as gpd

gdp_path = os.path.join(ROOT_DIR, "GDP_data")
admin_path = os.path.join(ROOT_DIR, "admin")
os.makedirs(gdp_path, exist_ok=True)

In [ ]:
# Download Kummu v4 30 arc-sec total GDP raster (1.4 GB - skips if already present)
tif_path = download_file(
    url=KUMMU_V4_FILES["rast_gdpTot_1990_2024_30arcsec.tif"],
    out_dir=gdp_path,
)
print("Source raster:", tif_path)

## Strategy 1 - Windowed extraction (requires local boundary shapefile)

In [ ]:
# Load Nigeria boundary (ADM2 shapefile already downloaded)
nga_gdf = gpd.read_file(os.path.join(admin_path, "Nigeria_admin2.shp"))

da_windowed = preprocess_kummu_gdp_tif_windowed(
    tif_path=tif_path,
    year=2024,
    boundary_gdf=nga_gdf,
    out_path=os.path.join(gdp_path, "GDP_PPP_2024_NGA_windowed.tif"),
)
print("Total Nigeria GDP 2024 (windowed):", float(da_windowed.sum()))

## Strategy 2 - General extractor (auto-downloads boundary from geoBoundaries API)

In [ ]:
da = extract_country_gdp(
    country="NGA",
    year=2024,
    gdp_path=gdp_path,
    out_path=os.path.join(gdp_path, "GDP_PPP_2024_NGA.tif"),
)
print("Total Nigeria GDP 2024 (general extractor):", float(da.sum()))